# Burmese animal text quality audit

Read-only audit of `wildlife.db`. No animal data is changed.

**Scope:** 123 animal records, 28 published Burmese or Burmese-facing text fields, 1,843 non-empty cells. Internal provenance JSON, URLs, and image fields are excluded.


## Summary

- 24 animals contain high-confidence visible spelling or repetition issues.
- 58 animals contain legacy/non-standard `ဥ်/ဉ်`, `ဦး/ဦး`, or `အဆောက်အဦး/အဆောက်အဦ` spellings.
- 4 visually normalized Burmese common names are shared by two different scientific species.
- The audit is intentionally conservative: ambiguous zoological common names are not treated as confirmed typos.


In [ ]:
import re, sqlite3, collections

db = sqlite3.connect('file:../wildlife.db?mode=ro', uri=True)
db.row_factory = sqlite3.Row
columns = [row[1] for row in db.execute('PRAGMA table_info(animals)')]
fields = list(dict.fromkeys(
    ['name_my'] + [c for c in columns if c.endswith('_my')] +
    ['conservation_status', 'population_trend', 'myanmar_population',
     'global_population', 'other_name']
))
animals = db.execute(
    'SELECT ' + ','.join(['id', 'name_en', 'scientific_name'] + fields) +
    ' FROM animals ORDER BY id'
).fetchall()

len(animals), len(fields), sum(
    bool(str(row[field]).strip())
    for row in animals for field in fields if row[field] is not None
)


## High-confidence findings

Animal IDs: `1, 6, 8, 10, 13, 14, 16, 18, 20, 24, 25, 31, 32, 35, 36, 39, 40, 41, 46, 57, 63, 67, 71, 80`.

Confirmed repetition is present in IDs `1, 14, 25, 31, 35, 57, 71, 80`. Other IDs in the set contain high-confidence visible word or character errors.


In [ ]:
high_confidence_patterns = {
    'ဖြစ်ပေါ််': 'ဖြစ်ပေါ်',
    'အမျိုးသာား': 'အမျိုးသား',
    'အကောာင်ရေ': 'အကောင်ရေ',
    'တစ်ကောင်ထီးထီးး': 'တစ်ကောင်ထီးထီး',
    'ဖော််ပြ': 'ဖော်ပြ',
    'သသတ္တဝါ': 'သတ္တဝါ',
    '၎င်းသ်': '၎င်းသည်',
    'ခြုံလှောင်': 'ခြံလှောင်',
    '၎င်လိပ်မျိုးများ': '၎င်းလိပ်မျိုးများ',
    'မန္တာလေး': 'မန္တလေး',
    'အနှောက်အယှက်': 'အနှောင့်အယှက်',
    'ထိန်းသိမ်ရေး': 'ထိန်းသိမ်းရေး',
    'အန္တာရာယ်': 'အန္တရာယ်',
    'လိပ်ခွံအတ္ငင်း': 'လိပ်ခွံအတွင်း',
    'ပုံးနေလျက်': 'ပုန်းနေလျက်',
    'ပြန်လွှန်ပေး': 'ပြန်လွှတ်ပေး',
    'ဥပဒေစိုးမိုးရေ': 'ဥပဒေစိုးမိုးရေး',
    'အထူးအဖြင့်': 'အထူးသဖြင့်',
    'တိရိစ္ဆာန်': 'တိရစ္ဆာန်',
    'ဖော်ပြန်ပြောင်းလဲ': 'ဖောက်ပြန်ပြောင်းလဲ',
    'ဆ်မြောင်း': 'ဆည်မြောင်း',
    'စိုက်မျိုးမြေ': 'စိုက်ပျိုးမြေ',
    'လူလား': 'လူသား',
    'လူနေအိပ်ယာ': 'လူနေအိမ်ရာ',
    'ရှမ်းပြန်နယ်': 'ရှမ်းပြည်နယ်',
    'ကျယ်ကျယ်ပြန်ပြန့်': 'ကျယ်ကျယ်ပြန့်ပြန့်',
    'မတူညီသေ': 'မတူညီသော',
    'လျော့ကျ့ကျနှုန်း': 'လျော့ကျနှုန်း',
    'တရားမတင် ပိုက်ဆွဲ': 'တရားမဝင် ပိုက်ဆွဲ',
    'ကြိုးကြားခေါင်းနီ': 'ကြိုးကြာခေါင်းနီ',
    'ထိန်းသိမ်‌ရေး': 'ထိန်းသိမ်းရေး',
    'မကြာခန': 'မကြာခဏ',
    'တစ်ခုမှာ တစ်ခုမှာ': 'တစ်ခုမှာ',
    'စိမ့်များ၊ စိမ့်များ': 'စိမ့်များ',
    'ဥ ဥချတတ်': 'ဥချတတ်',
    'အပူပိုင်း အပူပိုင်းရေပြင်': 'အပူပိုင်းရေပြင်',
}


## Interpretation notes

The `ဥ်/ဉ်` and `ဦး/ဦး` group is kept separate because these forms often come from legacy source text or inconsistent Unicode practice. Duplicate common names are also kept separate because two species can legitimately share a broad vernacular name; they require domain review rather than automatic correction.
